In [1]:
import pandas as pd
import ast

In [2]:
df = pd.read_csv("Formula1_Pitstop_Data_1950-2024_all_rounds.csv")
df.head()

,Season,Round,Circuit,Driver,Constructor,Laps,Position,TotalPitStops,AvgPitStopTime,PitStops
0,1950,1,Silverstone Circuit,Nino Farina,Alfa Romeo,70,1,0,NaN,[]
1,1950,1,Silverstone Circuit,Luigi Fagioli,Alfa Romeo,70,2,0,NaN,[]
2,1950,1,Silverstone Circuit,Reg Parnell,Alfa Romeo,70,3,0,NaN,[]
3,1950,1,Silverstone Circuit,Yves Cabantous,Talbot-Lago,68,4,0,NaN,[]
4,1950,1,Silverstone Circuit,Louis Rosier,Talbot-Lago,68,5,0,NaN,[]


In [3]:
df["PitStops"].iloc[0]

'[]'

In [4]:
def parse_pitstops(cell):
    if pd.isna(cell) or cell == "[]" or cell == []:
        return []
    try:
        return ast.literal_eval(cell)  # safely evaluate string lists
    except:
        return []


In [5]:
df["PitStopsParsed"] = df["PitStops"].apply(parse_pitstops)
df["PitStopsParsed"].head()


0    []
1    []
2    []
3    []
4    []
Name: PitStopsParsed, dtype: object

In [6]:
rows = []
for _, row in df.iterrows():
    for stop in row["PitStopsParsed"]:
        rows.append({
            "Season": row["Season"],
            "Round": row["Round"],
            "Circuit": row["Circuit"],
            "Driver": row["Driver"],
            "Constructor": row["Constructor"],
            "LapsInRace": row["Laps"],
            "Position": row["Position"],
            "PitLap": stop.get("Lap"),
            "PitStopTime_s": stop.get("StopTime")
        })

pit = pd.DataFrame(rows)
pit.head()


,Season,Round,Circuit,Driver,Constructor,LapsInRace,Position,PitLap,PitStopTime_s
0,2011,1,Albert Park Grand Prix Circuit,Sebastian Vettel,Red Bull,58,1,14,22.603
1,2011,1,Albert Park Grand Prix Circuit,Sebastian Vettel,Red Bull,58,1,36,24.036
2,2011,1,Albert Park Grand Prix Circuit,Lewis Hamilton,McLaren,58,2,16,23.227
3,2011,1,Albert Park Grand Prix Circuit,Lewis Hamilton,McLaren,58,2,36,23.199
4,2011,1,Albert Park Grand Prix Circuit,Vitaly Petrov,Renault,58,3,16,24.535


In [7]:
before = len(pit)
pit = pit.dropna(subset=["PitLap", "PitStopTime_s"])
pit = pit[(pit["PitStopTime_s"] > 0) & (pit["PitStopTime_s"] < 60)]
after = len(pit)
print(before, after)


11370 10853


In [8]:
pit.to_csv("f1_pitstops_clean_long.csv", index=False)


In [9]:
# Fastest pit stop per team per season
pit["FastestTeamStop"] = pit.groupby(["Season", "Constructor"])["PitStopTime_s"].transform("min")

# Delta from fastest (lower is better)
pit["DeltaFromFastest"] = pit["PitStopTime_s"] - pit["FastestTeamStop"]

# Team average
pit["TeamAvg"] = pit.groupby("Constructor")["PitStopTime_s"].transform("mean")

# Consistency score (lower stdev means more consistent)
pit["TeamStd"] = pit.groupby("Constructor")["PitStopTime_s"].transform("std")
